# Lab 4.1 — Text-Attributed Graphs: LLM Features for GNNs
**Module IV · LLMs + GNNs Together**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_labs/blob/main/module-4-hybrid/lab4_1_text_attributed_graphs.ipynb)

---

## What you will do
1. Build a **text-attributed graph** — a graph where each node is a document with a text description.
2. Compare two types of node features: **TF-IDF** (traditional bag-of-words) vs. **sentence-transformer embeddings** (from Module II).
3. Train MLP and GCN models on both feature types and compare their accuracy in a **low-data regime**.
4. Use **t-SNE** to visualise how well each feature type separates the classes — before and after GCN message passing.
5. Understand why **LLM features + graph structure** is the most powerful combination.

## The core question
In Module III, Cora's features were 1,433-dimensional sparse binary bag-of-words — a crude representation of paper content. What if we replaced them with dense 384-dimensional embeddings from a language model? And does the graph still help on top of good LLM features?

## Prerequisites
Labs 2.3 (sentence-transformers, FAISS) and 3.2 (GCN training) completed.

**Estimated time:** 55–70 min

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_labs.git",
         "labs_assignment"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         "labs_assignment/environment/requirements.txt"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)
    sys.path.insert(0, "labs_assignment")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

from utils import plot_embeddings, check_graph, check_gnn_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

CLASS_NAMES = ["Graph Learning", "Language Models", "Computer Vision", "Optimisation"]

---
## 1 · A mini text-attributed citation graph

We work with a self-contained dataset of **48 research papers** spanning 4 topics (12 papers each). Each paper has a title, a two-sentence abstract, and a class label. Papers on the same topic are more likely to cite each other.

This is a tiny, controlled version of the Cora dataset — but with **actual text** that we can embed with a language model. The key question: does replacing Cora-style bag-of-words with LLM embeddings improve learning, and does the graph still help on top?

In [ ]:
# Paper dataset: 48 papers × 4 topics (12 per topic)
_raw = [
    # ── Class 0: Graph Learning ─────────────────────────────────────────────
    ("Graph Convolutional Networks for Semi-Supervised Classification",
     "Graph convolutional networks learn node representations by aggregating normalised neighbour features via spectral graph filters. The model achieves state-of-the-art semi-supervised node classification with very few training labels.", 0),
    ("Graph Attention Networks",
     "GATs replace fixed degree-normalised aggregation with learnable per-edge attention coefficients. Multi-head attention allows each node to focus on its most informative neighbours adaptively.", 0),
    ("Inductive Representation Learning with GraphSAGE",
     "GraphSAGE samples and aggregates local neighbourhood features to learn inductive node embeddings. The model generalises to unseen nodes at inference time, enabling deployment on dynamic and large-scale graphs.", 0),
    ("How Powerful are Graph Neural Networks?",
     "We characterise GNN expressive power in terms of the Weisfeiler-Lehman graph isomorphism test and design maximally powerful GINs. Most popular GNN architectures are provably less discriminative than the WL limit.", 0),
    ("Over-Smoothing in Deep Graph Neural Networks",
     "Stacking many GNN layers causes node representations to converge to indistinguishable vectors in a phenomenon called over-smoothing. This explains why shallow two-layer GNNs empirically outperform deeper variants on node classification benchmarks.", 0),
    ("Graph Transformers with Structural Positional Encodings",
     "Graph Transformers apply global self-attention conditioned on structural encodings derived from random walks and Laplacian eigenvectors. The architecture captures long-range dependencies that local message-passing GNNs fundamentally cannot model.", 0),
    ("Hierarchical Differentiable Graph Pooling",
     "DiffPool learns hierarchical graph representations via differentiable soft node clustering applied at each layer. The method enables end-to-end graph classification requiring multi-scale structural understanding.", 0),
    ("Deep Graph Infomax for Unsupervised Node Embeddings",
     "We train node encoders without labels by maximising mutual information between local node representations and a global graph summary. The self-supervised approach achieves performance competitive with supervised node classification on standard benchmarks.", 0),
    ("Over-Squashing and Graph Topology in GNNs",
     "We show that graph topology creates information bottlenecks in message-passing GNNs, limiting their ability to propagate long-range signals in a phenomenon called over-squashing. Graph rewiring strategies based on curvature can alleviate this structural limitation.", 0),
    ("Personalized PageRank for Graph Propagation",
     "APPNP decouples feature transformation from propagation using personalised PageRank to spread label information across the graph. The design avoids over-smoothing while enabling aggregation from distant neighbours beyond 2-hop receptive fields.", 0),
    ("Scalable Mini-Batch GNN Training with Node Sampling",
     "We propose neighbourhood sampling strategies to enable mini-batch training on graphs with millions of nodes. Unbiased gradient estimators ensure the mini-batch training converges to the same solution as full-graph training.", 0),
    ("Relational Deep Learning on Enterprise Database Graphs",
     "Relational deep learning extends GNNs to heterogeneous multi-table relational databases by constructing entity graphs from foreign-key relationships. The approach enables end-to-end learning on structured enterprise data without manual feature engineering.", 0),
    # ── Class 1: Language Models ─────────────────────────────────────────────
    ("Attention Is All You Need",
     "We introduce the Transformer, a sequence model based entirely on self-attention mechanisms without recurrence or convolutions. Transformers train efficiently in parallel and achieve state-of-the-art translation quality with better scaling than recurrent architectures.", 1),
    ("BERT: Pre-Training Deep Bidirectional Transformers",
     "BERT pre-trains deep bidirectional representations by jointly predicting masked tokens and next-sentence relationships. Fine-tuning BERT with a single task-specific output layer achieves state-of-the-art results on eleven diverse natural language benchmarks.", 1),
    ("GPT-3: Language Models are Few-Shot Learners",
     "We train a 175-billion-parameter autoregressive language model that performs competitive in-context learning from a handful of examples. Emergent few-shot capabilities suggest that scale qualitatively changes language model behaviour and reasoning ability.", 1),
    ("Retrieval-Augmented Generation for Knowledge-Intensive NLP",
     "RAG combines a parametric sequence-to-sequence model with non-parametric dense retrieval over an external document index. Grounding generation in retrieved evidence reduces hallucination and enables knowledge updates without expensive model retraining.", 1),
    ("Sentence-BERT for Semantic Similarity Search",
     "We fine-tune BERT with siamese network training to produce fixed-size sentence embeddings optimised for semantic similarity. The resulting embeddings enable fast semantic search and clustering at scales impractical for cross-encoder rerankers.", 1),
    ("Finetuned Language Models as Zero-Shot Learners",
     "Instruction-tuning on diverse NLP tasks described in natural language enables zero-shot generalisation to completely unseen tasks. Task diversity and training scale are both critical factors for strong instruction-following behaviour.", 1),
    ("Training Language Models to Follow Instructions with Human Feedback",
     "InstructGPT fine-tunes GPT-3 using reinforcement learning from human preference rankings to align model outputs with user intent. Human evaluators consistently prefer aligned models despite them being much smaller than the base GPT-3.", 1),
    ("Hallucination in Large Language Models: A Survey",
     "LLMs generate factually incorrect statements with high confidence, a structural consequence of next-token prediction optimisation rather than factual grounding. Retrieval augmentation and factuality fine-tuning are among the most effective mitigation strategies identified.", 1),
    ("Constitutional AI: Harmlessness from AI Feedback",
     "We train harmless AI assistants using a constitutional principles framework without requiring human harmlessness labels for each training example. AI-generated critiques and revisions guided by stated principles enable scalable safety alignment.", 1),
    ("Scaling Laws for Neural Language Models",
     "Language model test loss follows precise power-law scaling relationships with model size, training data size, and compute budget. Optimal allocation of a fixed compute budget strongly favours large models trained on proportionally less data.", 1),
    ("Quantisation Methods for Efficient LLM Inference",
     "Large language model weights can be compressed to 4-bit integer precision with negligible accuracy degradation using post-training quantisation. Quantisation reduces GPU memory requirements by 4-8x and enables large-model inference on consumer hardware.", 1),
    ("Chain-of-Thought Prompting Elicits Reasoning in LLMs",
     "Prompting language models to produce step-by-step reasoning chains before final answers substantially improves multi-step arithmetic and commonsense benchmarks. The capability emerges abruptly in models above approximately 100 billion parameters.", 1),
    # ── Class 2: Computer Vision ─────────────────────────────────────────────
    ("Deep Residual Networks for Image Recognition",
     "Residual networks introduce identity shortcut connections that enable training of networks over 100 layers deep without degradation. ResNet-152 won the ImageNet 2015 competition with 3.57% top-5 error, setting a new paradigm for deep visual recognition.", 2),
    ("An Image Is Worth 16x16 Words: Vision Transformer",
     "We apply the standard Transformer directly to non-overlapping image patches treated as token sequences for image classification. With large-scale pre-training, Vision Transformers outperform convolutional networks and require fewer architectural inductive biases.", 2),
    ("Learning Visual Representations from Natural Language with CLIP",
     "CLIP trains image and text encoders jointly via contrastive learning on 400 million web-scraped image-caption pairs. The model enables zero-shot image classification competitive with fully supervised models across thirty diverse downstream datasets.", 2),
    ("Denoising Diffusion Probabilistic Models",
     "Diffusion models generate high-quality images by learning to reverse a gradual Gaussian noising process through denoising score matching. They surpass GANs in sample quality and diversity on standard image generation benchmarks without adversarial training instabilities.", 2),
    ("Masked Autoencoders as Scalable Vision Learners",
     "Masking 75% of image patches and training a Vision Transformer to reconstruct the missing pixels yields powerful self-supervised visual representations. MAE outperforms contrastive self-supervised approaches and scales efficiently to very large architectures.", 2),
    ("Generative Adversarial Networks",
     "GANs train a generator and discriminator in a minimax adversarial game, with the generator learning to produce images indistinguishable from real training data. The framework enables high-fidelity image synthesis without requiring explicit density estimation or variational inference.", 2),
    ("You Only Look Once: Unified Real-Time Object Detection",
     "YOLO frames object detection as a regression problem predicting bounding boxes and class probabilities from full images in a single forward pass. The unified architecture detects objects in real time with accuracy competitive with two-stage region proposal methods.", 2),
    ("Batch Normalisation for Accelerating Deep Network Training",
     "Batch normalisation normalises layer pre-activations by mini-batch statistics, acting as an adaptive conditioning technique that reduces internal covariate shift. It enables much higher learning rates, acts as implicit regularisation, and substantially accelerates deep network training.", 2),
    ("Swin Transformer: Hierarchical Vision Transformer with Shifted Windows",
     "The Swin Transformer introduces hierarchical feature maps and shifted-window self-attention to create a versatile vision backbone combining Transformer expressiveness with CNN locality. It achieves state-of-the-art on image classification, object detection, and dense prediction simultaneously.", 2),
    ("Contrastive Representation Learning with SimCLR",
     "SimCLR maximises cosine similarity between differently augmented views of the same image using a contrastive loss and a nonlinear projection head. Strong augmentation policies and large batch sizes are critical for achieving state-of-the-art unsupervised visual representations.", 2),
    ("Feature Pyramid Networks for Multi-Scale Object Detection",
     "FPN exploits the inherent multi-scale hierarchy of deep CNNs by creating a top-down pathway with lateral connections that fuse high-level semantics with high-resolution features. The architecture significantly improves small object detection accuracy across standard benchmarks.", 2),
    ("Neural Style Transfer Using Deep Convolutional Feature Correlations",
     "Neural style transfer optimises a content image to match the feature Gram matrix statistics of a reference artwork captured by a pretrained CNN. The technique decouples content and style representations, enabling high-quality artistic stylisation without explicit style models.", 2),
    # ── Class 3: Optimisation Methods ────────────────────────────────────────
    ("Adam: Adaptive Moment Estimation for Deep Learning",
     "Adam maintains exponential moving averages of gradients and squared gradients to compute per-parameter adaptive learning rates. The optimiser is robust to hyperparameter choice and has become the dominant method for training large neural networks across all domains.", 3),
    ("Dropout: Preventing Overfitting Through Stochastic Regularisation",
     "Dropout randomly deactivates neural network units during training to prevent co-adaptation of features, approximating an ensemble of exponentially many networks. The technique substantially reduces overfitting across all architecture types with negligible additional computation.", 3),
    ("Decoupled Weight Decay Regularisation with AdamW",
     "AdamW corrects Adam by decoupling L2 weight decay from the adaptive gradient update, fixing a subtle but consequential flaw in the original formulation. The correction consistently improves fine-tuning of large pre-trained Transformer models on downstream tasks.", 3),
    ("Layer Normalisation for Stable Transformer Training",
     "Layer normalisation normalises activations across the feature dimension within each training example, independently of batch size. Unlike batch normalisation, it is effective for sequence-to-sequence models and is now the standard normalisation layer in Transformer architectures.", 3),
    ("Gradient Clipping for Preventing Exploding Gradients",
     "Gradient clipping rescales the global gradient norm when it exceeds a threshold, preventing catastrophic parameter updates during training. Proper gradient clipping is essential for stable optimisation of deep recurrent networks and Transformers on long sequences.", 3),
    ("Cyclical Learning Rates for Faster Neural Network Training",
     "Cyclical learning rates oscillate between bounds rather than monotonically decaying, enabling the optimiser to escape poor basins and explore flatter loss regions. The approach often reaches better solutions in fewer total iterations than standard step decay schedules.", 3),
    ("Sharpness-Aware Minimisation for Improved Generalisation",
     "SAM simultaneously minimises loss value and landscape sharpness by perturbing parameters toward the steepest ascent direction before each gradient step. Flat loss minima correlate strongly with better test generalisation across image classification, NLP, and graph learning tasks.", 3),
    ("Mixed Precision Training with Dynamic Loss Scaling",
     "Training with FP16 arithmetic while maintaining FP32 master weights reduces GPU memory requirements by approximately half and increases throughput on modern hardware. Dynamic loss scaling prevents underflow in small FP16 gradients, maintaining training accuracy.", 3),
    ("Stochastic Gradient Descent with Cosine Annealing and Warm Restarts",
     "SGDR periodically resets the cosine annealing schedule with warm restarts, enabling the optimiser to escape local minima and explore multiple basins. Warm restarts produce natural model ensembles from checkpoints at restart boundaries, improving final accuracy.", 3),
    ("Loss Surface Geometry of Overparameterised Neural Networks",
     "We analyse neural network loss surface geometry, showing that overparameterisation makes poor local minima exponentially rare compared to saddle points. These theoretical insights help explain the empirical success of gradient-based training in overparameterised deep learning models.", 3),
    ("The Lookahead Optimiser for Stable Exploration",
     "Lookahead maintains fast inner and slow outer weight trajectories, performing a slow interpolation step every k fast updates to stabilise convergence. The optimiser consistently outperforms its base inner optimiser with minimal additional memory overhead.", 3),
    ("Learning Rate Warmup Schedules for Transformer Training",
     "Learning rate warmup prevents unstable early gradient updates by linearly increasing the rate from zero over the first few thousand steps. Combining warmup with cosine or inverse-square-root decay is now standard practice for training large Transformer models from scratch.", 3),
]

titles     = [r[0] for r in _raw]
abstracts  = [r[1] for r in _raw]
labels_np  = np.array([r[2] for r in _raw])

N = len(_raw)  # 48
print(f"Dataset: {N} papers across {len(CLASS_NAMES)} topics")
for c, name in enumerate(CLASS_NAMES):
    print(f"  Class {c} ({name}): {(labels_np == c).sum()} papers")

In [ ]:
# Build a synthetic citation graph using a stochastic block model
# Papers in the same topic cite each other with high probability;
# cross-topic citations exist but are much rarer.
rng = np.random.default_rng(42)

P_IN  = 0.35  # within-topic connection probability
P_OUT = 0.04  # cross-topic connection probability

edges_src, edges_dst = [], []
for i in range(N):
    for j in range(i + 1, N):
        prob = P_IN if labels_np[i] == labels_np[j] else P_OUT
        if rng.random() < prob:
            edges_src.extend([i, j])
            edges_dst.extend([j, i])  # undirected: both directions

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
y = torch.tensor(labels_np, dtype=torch.long)

# Split masks: 2 train / 5 val / 5 test per class (low-data regime)
N_PER_CLASS = 12
train_mask = torch.zeros(N, dtype=torch.bool)
val_mask   = torch.zeros(N, dtype=torch.bool)
test_mask  = torch.zeros(N, dtype=torch.bool)

for c in range(len(CLASS_NAMES)):
    start = c * N_PER_CLASS
    train_mask[start:start + 2]  = True  # 2 per class = 8 total
    val_mask[start + 2:start + 7]  = True  # 5 per class = 20 total
    test_mask[start + 7:start + 12] = True  # 5 per class = 20 total

print(f"Graph: {N} nodes, {edge_index.shape[1] // 2} undirected edges")
print(f"Splits — train: {train_mask.sum().item()}, "
      f"val: {val_mask.sum().item()}, test: {test_mask.sum().item()}")
print(f"Note: only {train_mask.sum().item()} labelled training nodes "
      f"for {N} total — intentionally sparse, like Cora!")

### Exercise 4.1.1 `[Core]` — Explore the graph

1. Convert the edge_index to a NetworkX graph and call `check_graph`.
2. Compute and print the mean degree.
3. What fraction of edges are within the same topic (homophily ratio)?

In [ ]:
# YOUR CODE HERE
# Hint: 1. Convert the edge_index to a NetworkX graph and call check_graph.
raise NotImplementedError("Complete this exercise")

> **Observation:** The graph has high homophily (~0.70–0.80) — most edges connect papers in the same research topic. This is realistic: papers cite related work more than unrelated work. High homophily means graph structure is highly informative for classification: a node is likely the same class as its neighbours. This is precisely the property that GCN exploits.

---
## 2 · TF-IDF node features — the traditional baseline

**TF-IDF** (Term Frequency-Inverse Document Frequency) represents each document as a sparse vector where each dimension corresponds to a vocabulary word and the value reflects how distinctive that word is for this document.

This is essentially what Cora's 1,433-dimensional features are: a bag-of-words over paper abstracts. TF-IDF is a strong classical baseline for text classification.

### Exercise 4.1.2 `[Core]` — Build TF-IDF features and a PyG Data object

1. Use `sklearn.feature_extraction.text.TfidfVectorizer` with `max_features=200` to vectorise the 48 abstracts.
2. Convert the resulting matrix to a dense float32 NumPy array.
3. Create a PyG `Data` object with `x` (TF-IDF features), `edge_index`, `y`, and all three masks.
4. Print the feature shape.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Use sklearn.feature_extraction.text.TfidfVectorizer with max_features=200 to vectorise the 48 abstracts.
raise NotImplementedError("Complete this exercise")

---
## 3 · MLP vs GCN — does the graph help with sparse features?

We train two models on TF-IDF features:
- **MLP**: each paper classified independently from its TF-IDF vector — no graph.
- **GCN**: same capacity, but uses citation edges to aggregate neighbour features.

With only **2 training examples per class (8 total)**, we are in a very low-data regime. The graph provides crucial additional signal: if my neighbours are labelled, I can borrow from their labels.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_channels, hidden, out_channels, dropout=0.3):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden)
        self.lin2 = nn.Linear(hidden, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin2(x)


class GCN(nn.Module):
    def __init__(self, in_channels, hidden, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


def train_and_evaluate(model, data, n_epochs=500, lr=0.01, wd=5e-3, verbose=False):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for epoch in range(1, n_epochs + 1):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        F.cross_entropy(out[data.train_mask], data.y[data.train_mask]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        results = {split: (pred[getattr(data, f"{split}_mask")] ==
                           data.y[getattr(data, f"{split}_mask")]).float().mean().item()
                   for split in ["train", "val", "test"]}
    return results


print("Models and training function defined.")

### Exercise 4.1.3 `[Core]` — Train MLP and GCN on TF-IDF features

Train both models for 500 epochs on `data_tfidf`. Use `hidden=32`, `dropout=0.3`, `lr=0.01`, `weight_decay=5e-3`. Record and print test accuracy for each.

In [ ]:
# YOUR CODE HERE
# Hint: Train both models for 500 epochs on data_tfidf. Use hidden=32, dropout=0.3, lr=0.01, weight_decay=5e-3. Record and pr...
raise NotImplementedError("Complete this exercise")

---
## 4 · Sentence-transformer node features — LLM embeddings

Now we swap the sparse TF-IDF features for dense embeddings from `all-MiniLM-L6-v2` — the same model you used in Lab 2.3 for RAG. Instead of counting words, the language model encodes the *meaning* of each abstract into a 384-dimensional vector.

Key difference:
| | TF-IDF | Sentence-Transformer |
|---|---|---|
| Dimension | 200 (sparse) | 384 (dense) |
| Captures | Word overlap | Semantic meaning |
| "Graph Networks" ≈ "GNN" | ✗ | ✓ (synonyms) |
| Training required | No | Pretrained LLM |

In [ ]:
# Provided — load the sentence-transformer (same model as Lab 2.3)
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Sentence-transformer loaded. Embedding dimension: {st_model.get_sentence_embedding_dimension()}")

### Exercise 4.1.4 `[Core]` — Encode abstracts and build a PyG Data object

1. Encode all 48 abstracts using `st_model.encode(abstracts, show_progress_bar=True)`.
2. Convert to float32 tensor.
3. Create `data_sent` — a PyG `Data` object with sentence-transformer features.
4. Print the feature shape and compare to TF-IDF.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Encode all 48 abstracts using st_model.encode(abstracts, show_progress_bar=True).
raise NotImplementedError("Complete this exercise")

### Exercise 4.1.5 `[Core]` — Train MLP and GCN on sentence-transformer features

Train both models on `data_sent`. Use the same hyperparameters and 500 epochs. Record results.

Then call `check_gnn_model("4.1.5", gcn_sent, data_sent, split="test", min_acc=0.88)`.

In [ ]:
# YOUR CODE HERE
# Hint: Train both models on data_sent. Use the same hyperparameters and 500 epochs. Record results.
raise NotImplementedError("Complete this exercise")

---
## 5 · Comparison across all four models

### Exercise 4.1.6 `[Core]` — Plot the comparison

Build a bar chart comparing the test accuracy of all four models. Add a dashed baseline at 0.25 (random chance with 4 classes). What can you conclude about the relative contribution of (a) better features and (b) graph structure?

In [ ]:
# YOUR CODE HERE
# Hint: Build a bar chart comparing the test accuracy of all four models. Add a dashed baseline at 0.25 (random chance with 4...
raise NotImplementedError("Complete this exercise")

> **Key results:** Both feature quality (LLM > TF-IDF) and graph structure (GCN > MLP) contribute positively, and they **compound**. With only 8 training nodes, sentence-transformer features dramatically outperform TF-IDF — the LLM has learned semantic representations that generalise from very few examples. The GCN then adds value on top by propagating label information through the citation graph. The best result comes from combining both: rich LLM features + structure-aware GNN.

---
## 6 · Visualising feature quality with t-SNE

The accuracy comparison tells us *what* happens. The t-SNE projection tells us *why*: we can see whether the chosen features already form clean, separable clusters in high-dimensional space — before any GCN training.

### Exercise 4.1.7 `[Core]` — t-SNE projection of raw features

Project TF-IDF features and sentence-transformer features to 2D using `plot_embeddings` (from the `utils` package) and compare them side by side. Which feature type forms cleaner clusters?

In [ ]:
# YOUR CODE HERE
# Hint: Project TF-IDF features and sentence-transformer features to 2D using plot_embeddings (from the utils package) and co...
raise NotImplementedError("Complete this exercise")

> **Key observation:** Sentence-transformer embeddings form tight, well-separated clusters *before any training* — the LLM has already done much of the work. TF-IDF features overlap more because different topics share vocabulary ('learning', 'training', 'model'). This visualisation explains why the MLP + SentTrans outperforms MLP + TF-IDF: better input representations make the classifier's job easier.

### Exercise 4.1.8 `[Extension]` — GCN embedding quality after training

Extract the **intermediate embeddings** (after the first GCN layer, before the second) from `gcn_tfidf` and `gcn_sent`. Project both to 2D and compare them to the raw feature projections. Does GCN message passing improve cluster quality over the raw features?

In [ ]:
# YOUR CODE HERE
# Hint: Extract the intermediate embeddings (after the first GCN layer, before the second) from gcn_tfidf and gcn_sent. Proje...
raise NotImplementedError("Complete this exercise")

---
## Summary

| Model | Features | Test acc | Lesson |
|---|---|---|---|
| MLP | TF-IDF | ~60-75% | Bag-of-words alone is weak with few labels |
| GCN | TF-IDF | ~75-85% | Graph structure rescues weak features |
| MLP | SentTrans | ~80-90% | LLM embeddings generalise well from few examples |
| GCN | SentTrans | ~90-100% | **Best: rich features + graph both contribute** |

**The bridge principle of Module IV:** The best systems combine the representational richness of LLMs with the structural reasoning of GNNs. Neither alone reaches the full potential.

**Next → Lab 4.2:** We build a **GraphRAG** system on the TechRetail knowledge base — using graph structure to retrieve richer context and answer multi-hop questions that stumped the basic RAG from Lab 2.3.